In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!pip install -q openai-whisper tqdm pandas

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Fri Nov 20 19:26:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             28W /   70W |    3081MiB /  15360MiB |      0%      Default |
|          

In [ ]:
import zipfile
from pathlib import Path

# === AJUSTE O CAMINHO DO SEU .ZIP E ONDE SALVAR O CSV NO DRIVE ===
CAMINHO_ZIP = Path("/content/drive/MyDrive/dataset_anotado.zip")  # <-- Ajuste aqui
PASTA_SAIDA_DRIVE = Path("/content/drive/MyDrive")                   # <-- Onde o CSV será salvo
ARQUIVO_SAIDA = PASTA_SAIDA_DRIVE / "transcrições.csv"

# Pasta temporária no SSD local do Colab (leitura muito mais rápida que no Drive)
PASTA_EXTRACAO = Path("/content/dataset_extraido")

assert CAMINHO_ZIP.exists(), f"Arquivo .zip não encontrado em: {CAMINHO_ZIP}"

# Extrai apenas se ainda não tiver sido extraído nesta sessão
if not PASTA_EXTRACAO.exists():
    print(f"Extraindo '{CAMINHO_ZIP.name}' para o disco local do Colab...")
    with zipfile.ZipFile(CAMINHO_ZIP, "r") as zip_ref:
        zip_ref.extractall(PASTA_EXTRACAO)
    print("Extração concluída!")
else:
    print("Dataset já extraído localmente nesta sessão. Pulando extração...")

# Busca automaticamente a pasta 'yes publi' independente da hierarquia interna do .zip
candidatos = [
    p for p in PASTA_EXTRACAO.rglob("*")
    if p.is_dir() and p.name.lower() in {"yes publi", "yes_publi"}
]

assert len(candidatos) > 0, f"Pasta 'yes publi' não encontrada dentro de {CAMINHO_ZIP.name}"
PASTA_YES_PUBLI = candidatos[0]

# === PARÂMETROS DO WHISPER ===
MODELO_WHISPER = "large-v3"
IDIOMA = "en"
INTERVALO_CHECKPOINT = 10

print(f"Pasta alvo localizada: {PASTA_YES_PUBLI}")
print(f"Checkpoints e CSV final serão salvos no Drive em: {ARQUIVO_SAIDA}")

Extraindo 'dataset_anotado.zip' para o disco local do Colab...
Extração concluída!
Pasta alvo localizada: /content/dataset_extraido/dataset_anotado/yes publi
Checkpoints e CSV final serão salvos no Drive em: /content/drive/MyDrive/transcrições.csv


In [ ]:
import subprocess
import torch
import whisper
import pandas as pd
from tqdm.notebook import tqdm

# 1. Lista todos os vídeos dentro de "yes publi"
EXTENSOES_VIDEO = {".mp4", ".mov", ".mkv", ".avi", ".webm"}
videos = sorted([
    p for p in PASTA_YES_PUBLI.rglob("*")
    if p.suffix.lower() in EXTENSOES_VIDEO
])
print(f"Total de vídeos encontrados em 'yes publi': {len(videos)}")

# 2. Carrega apenas registros válidos do checkpoint (descarta os 254 com erro anterior)
if ARQUIVO_SAIDA.exists():
    df_existente = pd.read_csv(ARQUIVO_SAIDA)
    df_validos = df_existente[~df_existente["status"].astype(str).str.startswith("erro")]
    processados = set(df_validos["caminho_relativo"].astype(str))
    registros = df_validos.to_dict("records")
    print(f"Registros válidos aproveitados: {len(processados)}. Vídeos a processar: {len(videos) - len(processados)}")
else:
    processados = set()
    registros = []

# 3. Inicializa o modelo na GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
usar_fp16 = (device == "cuda")
print(f"Carregando modelo '{MODELO_WHISPER}' em {device} (fp16={usar_fp16})...")

model = whisper.load_model(MODELO_WHISPER, device=device)

def obter_duracao(video_path):
    try:
        cmd = [
            "ffprobe", "-v", "error", "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1", str(video_path)
        ]
        out = subprocess.check_output(cmd, stderr=subprocess.DEVNULL).decode().strip()
        return round(float(out), 2)
    except Exception:
        return None

# 4. Loop principal de transcrição
novos_processados = 0

for video_path in tqdm(videos, desc="Transcrevendo vídeos"):
    rel_path = str(video_path.relative_to(PASTA_YES_PUBLI))

    if rel_path in processados:
        continue

    id_post = video_path.stem
    subpasta = video_path.parent.name if video_path.parent != PASTA_YES_PUBLI else "raiz"

    try:
        result = model.transcribe(
            str(video_path),
            language=IDIOMA,
            fp16=usar_fp16,
            condition_on_previous_text=False  # Evita loops de repetição em vídeos com música de fundo
        )

        texto_completo = result.get("text", "").strip()
        qtd_palavras = len(texto_completo.split()) if texto_completo else 0
        status = "sucesso" if qtd_palavras > 0 else "sem_fala_detectada"

        registros.append({
            "id_post": id_post,
            "arquivo": video_path.name,
            "subpasta_perfil": subpasta,
            "caminho_relativo": rel_path,
            "duracao_seg": obter_duracao(video_path),
            "idioma_detectado": result.get("language", IDIOMA),
            "qtd_palavras": qtd_palavras,
            "transcricao": texto_completo,
            "status": status
        })

    except Exception as e:
        msg_erro = f"erro: {type(e).__name__} - {e}"
        if novos_processados == 0:
            print(f"\nExemplo de erro no primeiro vídeo: {msg_erro}")
        registros.append({
            "id_post": id_post,
            "arquivo": video_path.name,
            "subpasta_perfil": subpasta,
            "caminho_relativo": rel_path,
            "duracao_seg": None,
            "idioma_detectado": None,
            "qtd_palavras": 0,
            "transcricao": "",
            "status": msg_erro
        })

    processados.add(rel_path)
    novos_processados += 1

    if novos_processados % INTERVALO_CHECKPOINT == 0:
        pd.DataFrame(registros).to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8-sig")

# Salvamento final
df_final = pd.DataFrame(registros)
df_final.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8-sig")
print(f"\nConcluído! {len(df_final)} registros salvos em: {ARQUIVO_SAIDA}")

Total de vídeos encontrados em 'yes publi': 254
Registros válidos aproveitados: 0. Vídeos a processar: 254
Carregando modelo 'large-v3' em cuda (fp16=True)...


100%|█████████████████████████████████████| 2.88G/2.88G [01:12<00:00, 42.6MiB/s]


Transcrevendo vídeos:   0%|          | 0/254 [00:00<?, ?it/s]


Concluído! 254 registros salvos em: /content/drive/MyDrive/transcrições.csv


In [ ]:
df = pd.read_csv(ARQUIVO_SAIDA)

print("=== RESUMO DA TRANSCRIÇÃO ===")
print(df["status"].value_counts())
print(f"\nMédia de palavras por vídeo (com fala): {df[df['qtd_palavras'] > 0]['qtd_palavras'].mean():.1f}")
print(f"Duração média dos vídeos: {df['duracao_seg'].mean():.1f}s")

print("\n=== AMOSTRA DAS PRIMEIRAS TRANSCRIÇÕES ===")
display(df[df["qtd_palavras"] > 0][["id_post", "subpasta_perfil", "duracao_seg", "qtd_palavras", "transcricao"]].head(10))

=== RESUMO DA TRANSCRIÇÃO ===
status
sucesso                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

,id_post,subpasta_perfil,duracao_seg,qtd_palavras,transcricao
0,1699506053810524285,raiz,21.45,1,🎵
1,1793020869384205875,raiz,60.23,54,Nothing brings me down We are partnered with a...
2,1797463378462089746,raiz,10.73,29,Oh no! E இ BA Bnię N Ä Ecchan and ell a Wei an...
3,1802630644489045682,raiz,60.14,134,Oh no! Axe has got a little surprise for you. ...
4,1806806600816568333,raiz,6.64,20,"Hey, come show off your dance moves with me at..."
5,1893776756022492091,raiz,18.25,1,🎵
6,1896497022993524425,raiz,27.42,21,"If you say you're taller than me, or if you ca..."
7,1939382998081611373,raiz,60.16,158,Hi. Today I got a gift from Samsung and I'm go...
8,2148862599840538232,raiz,15.37,20,You're a rich girl and you're going too far Ca...
9,2396147329821503481,raiz,60.19,147,Let's go! That is much bigger than I thought i...
